# TensorFlow版 パターン評価学習

添付の元コードに合わせて、`diagonal8` / `edge2X` / `triangle` のパターンをニューラルネットで学習します。

学習では TensorFlow / numpy / tqdm を使います。提出用 `MyPlayer` では、ここで出力した `NEURAL_MODEL` を貼り付け、標準ライブラリだけで同じ順伝播を計算します。


In [ ]:
# 必要なら実行
# !pip install tensorflow numpy tqdm


In [ ]:
from copy import deepcopy
import datetime
import json
import pathlib
import random
import subprocess

import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Add, Concatenate, Dense, Input, LeakyReLU
from tensorflow.keras.models import Model
from tqdm import trange, tqdm

RECORD_DIR = pathlib.Path("training_records")
EVALUATE_OUT = pathlib.Path("evaluate.out")
USE_EVALUATE_OUT = EVALUATE_OUT.exists()

HW = 8
HW2 = 64
VACANT = -1
BLACK = 0
WHITE = 1
DIRECTIONS = (
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1),
)

print("record dir:", RECORD_DIR)
print("use evaluate.out:", USE_EVALUATE_OUT)


In [ ]:
diagonal8_idx = [[0, 9, 18, 27, 36, 45, 54, 63], [7, 14, 21, 28, 35, 42, 49, 56]]
for pattern in deepcopy(diagonal8_idx):
    diagonal8_idx.append(list(reversed(pattern)))

edge_2x_idx = [
    [9, 0, 1, 2, 3, 4, 5, 6, 7, 14],
    [9, 0, 8, 16, 24, 32, 40, 48, 56, 49],
    [49, 56, 57, 58, 59, 60, 61, 62, 63, 54],
    [54, 63, 55, 47, 39, 31, 23, 15, 7, 14],
]
for pattern in deepcopy(edge_2x_idx):
    edge_2x_idx.append(list(reversed(pattern)))

triangle_idx = [
    [0, 1, 2, 3, 8, 9, 10, 16, 17, 24], [0, 8, 16, 24, 1, 9, 17, 2, 10, 3],
    [7, 6, 5, 4, 15, 14, 13, 23, 22, 31], [7, 15, 23, 31, 6, 14, 22, 5, 13, 4],
    [63, 62, 61, 60, 55, 54, 53, 47, 46, 39], [63, 55, 47, 39, 62, 54, 46, 61, 53, 60],
    [56, 57, 58, 59, 48, 49, 50, 40, 41, 32], [56, 48, 40, 32, 57, 49, 41, 58, 50, 59],
]

pattern_idx = [diagonal8_idx, edge_2x_idx, triangle_idx]
names = ["diagonal8", "edge2X", "triangle"]
ln_in = sum(len(elem) for elem in pattern_idx) + 1

print("patterns:", dict(zip(names, [len(elem) for elem in pattern_idx])))
print("inputs:", ln_in)


In [ ]:
def parse_record_line(line):
    line = line.strip().lower()
    if not line or line.startswith("#"):
        return []
    for ch in ",;":
        line = line.replace(ch, " ")
    parts = line.split()
    if len(parts) == 1:
        text = parts[0]
        if len(text) % 2 != 0:
            raise ValueError(f"odd-length record: {line[:40]}")
        parts = [text[i:i+2] for i in range(0, len(text), 2)]
    moves = []
    for token in parts:
        if len(token) != 2 or token[0] < "a" or token[0] > "h" or token[1] < "1" or token[1] > "8":
            raise ValueError(f"bad move token: {token}")
        row = int(token[1]) - 1
        col = ord(token[0]) - ord("a")
        moves.append((row, col))
    return moves

def initial_board():
    board = [[VACANT for _ in range(HW)] for _ in range(HW)]
    board[3][3], board[4][4] = WHITE, WHITE
    board[3][4], board[4][3] = BLACK, BLACK
    return board

def opponent(player):
    return WHITE if player == BLACK else BLACK

def flips_for(board, row, col, player):
    if board[row][col] != VACANT:
        return []
    other = opponent(player)
    flips = []
    for dr, dc in DIRECTIONS:
        r, c = row + dr, col + dc
        line = []
        while 0 <= r < HW and 0 <= c < HW and board[r][c] == other:
            line.append((r, c))
            r += dr
            c += dc
        if line and 0 <= r < HW and 0 <= c < HW and board[r][c] == player:
            flips.extend(line)
    return flips

def legal_moves(board, player):
    return [(r, c) for r in range(HW) for c in range(HW) if flips_for(board, r, c, player)]

def apply_move(board, move, player):
    row, col = move
    next_board = [line[:] for line in board]
    next_board[row][col] = player
    for r, c in flips_for(board, row, col, player):
        next_board[r][c] = player
    return next_board

def board_to_str(board):
    chars = []
    for row in board:
        for cell in row:
            if cell == VACANT:
                chars.append(".")
            else:
                chars.append(str(cell))
    return "".join(chars)

def final_result(board):
    black = sum(cell == BLACK for row in board for cell in row)
    white = sum(cell == WHITE for row in board for cell in row)
    result = black - white
    n_vacant = HW2 - black - white
    if result > 0:
        result += n_vacant
    elif result < 0:
        result -= n_vacant
    return result

def fallback_additional_params(board, player):
    # evaluate.out がない環境でも、提出用コードと同じ3入力を作る。
    other = opponent(player)
    mobility = len(legal_moves(board, player))
    other_mobility = len(legal_moves(board, other))
    corners = [(0, 0), (0, 7), (7, 0), (7, 7)]
    own_corners = sum(board[r][c] == player for r, c in corners)
    other_corners = sum(board[r][c] == other for r, c in corners)
    frontier = 0
    other_frontier = 0
    for r in range(HW):
        for c in range(HW):
            if board[r][c] == VACANT:
                continue
            touches_empty = any(
                0 <= r + dr < HW and 0 <= c + dc < HW and board[r + dr][c + dc] == VACANT
                for dr, dc in DIRECTIONS
            )
            if touches_empty:
                if board[r][c] == player:
                    frontier += 1
                else:
                    other_frontier += 1
    v1 = mobility
    v2 = 15 + 3 * (own_corners - other_corners)
    v3 = 15 + (other_frontier - frontier)
    return [float(v1), float(v2), float(v3)]


In [ ]:
def load_records(record_dir=RECORD_DIR, max_games=None):
    paths = sorted(record_dir.glob("*.txt"))
    records = []
    for path in paths:
        with path.open("r", encoding="utf-8") as file:
            for line_no, line in enumerate(file, 1):
                moves = parse_record_line(line)
                if moves:
                    records.append((path.name, line_no, moves))
                    if max_games is not None and len(records) >= max_games:
                        return records
    return records

def records_to_data(records, shuffle_games=True, seed=42):
    records = list(records)
    if shuffle_games:
        random.seed(seed)
        random.shuffle(records)

    data = []
    n_skipped = 0
    evaluator = None
    if USE_EVALUATE_OUT:
        evaluator = subprocess.Popen(str(EVALUATE_OUT).split(), stdin=subprocess.PIPE, stdout=subprocess.PIPE)

    try:
        for file_name, line_no, moves in tqdm(records):
            board = initial_board()
            player = BLACK
            board_data = []
            try:
                for move_no, move in enumerate(moves, 1):
                    board_str = board_to_str(board)
                    if evaluator is not None:
                        evaluator.stdin.write((str(player) + "\n" + board_str + "\n").encode("utf-8"))
                        evaluator.stdin.flush()
                        additional = [float(elem) for elem in evaluator.stdout.readline().decode().split()]
                    else:
                        additional = fallback_additional_params(board, player)
                    board_data.append([board_str, player, additional[0], additional[1], additional[2]])

                    if not legal_moves(board, player):
                        player = opponent(player)
                    if move not in legal_moves(board, player):
                        raise ValueError(f"illegal move {move} at {file_name}:{line_no}, move {move_no}, player {player}")
                    board = apply_move(board, move, player)
                    player = opponent(player)
            except ValueError as error:
                n_skipped += 1
                if n_skipped <= 5:
                    print("skip:", error)
                continue

            result = final_result(board)
            for board_datum in board_data:
                board_datum.append(result)
                data.append(board_datum)
    finally:
        if evaluator is not None:
            evaluator.kill()

    print("n_data", len(data), "skipped", n_skipped)
    return data

records = load_records()
print("loaded games:", len(records))
data = records_to_data(records, shuffle_games=True, seed=42) if records else []

In [ ]:
all_data = [[] for _ in range(ln_in)]
all_labels = []

def make_lines(board, patterns, player):
    res = []
    for pattern in patterns:
        tmp = []
        for elem in pattern:
            tmp.append(1.0 if board[elem] == str(player) else 0.0)
        for elem in pattern:
            tmp.append(1.0 if board[elem] == str(1 - player) else 0.0)
        res.append(tmp)
    return res

def collect_data(board, player, v1, v2, v3, result):
    global all_data, all_labels
    v1 = float(v1)
    v2 = float(v2)
    v3 = float(v3)
    result = float(result) / 64
    idx = 0
    for i in range(len(pattern_idx)):
        lines = make_lines(board, pattern_idx[i], 0)
        for line in lines:
            all_data[idx].append(line)
            idx += 1
    all_data[idx].append([v1 / 30, (v2 - 15) / 15, (v3 - 15) / 15])
    all_labels.append(result)

for i in trange(len(data)):
    collect_data(*data[i])

len_data = len(all_labels)
print("len_data", len_data)


In [ ]:
x = [None for _ in range(ln_in)]
ys = []
idx = 0
for i in range(len(pattern_idx)):
    layers = []
    layers.append(Dense(16, name=names[i] + "_dense0"))
    layers.append(LeakyReLU(alpha=0.01))
    layers.append(Dense(16, name=names[i] + "_dense1"))
    layers.append(LeakyReLU(alpha=0.01))
    layers.append(Dense(1, name=names[i] + "_out"))
    layers.append(LeakyReLU(alpha=0.01))
    add_elems = []
    for j in range(len(pattern_idx[i])):
        x[idx] = Input(shape=len(pattern_idx[i][0]) * 2, name=names[i] + "_in_" + str(j))
        tmp = x[idx]
        for layer in layers:
            tmp = layer(tmp)
        add_elems.append(tmp)
        idx += 1
    ys.append(Add()(add_elems))

y_pattern = Concatenate(axis=-1)(ys)
x[idx] = Input(shape=3, name="additional_input")
y_add = Dense(8, name="add_dense0")(x[idx])
y_add = LeakyReLU(alpha=0.01)(y_add)
y_add = Dense(1, name="add_dense1")(y_add)
y_add = LeakyReLU(alpha=0.01)(y_add)
y_all = Concatenate(axis=-1)([y_pattern, y_add])
y_all = Dense(1, name="all_dense0")(y_all)

model = Model(inputs=x, outputs=y_all)
model.summary()
model.compile(loss="mse", metrics="mae", optimizer="adam")


In [ ]:
test_ratio = 0.1
n_epochs = 10

if len_data == 0:
    raise RuntimeError("training_records/*.txt に棋譜を入れてから実行してください。")

shuffled = list(range(len_data))
random.shuffle(shuffled)
shuffled_data = [[arr[i] for i in shuffled] for arr in all_data]
shuffled_labels = [all_labels[i] for i in shuffled]

all_data_np = [np.array(arr) for arr in shuffled_data]
all_labels_np = np.array(shuffled_labels)

n_train_data = int(len_data * (1.0 - test_ratio))
train_data = [arr[:n_train_data] for arr in all_data_np]
train_labels = all_labels_np[:n_train_data]
test_data = [arr[n_train_data:] for arr in all_data_np]
test_labels = all_labels_np[n_train_data:]

print(model.evaluate(test_data, test_labels))
early_stop = EarlyStopping(monitor="val_loss", patience=5)
history = model.fit(
    train_data,
    train_labels,
    epochs=n_epochs,
    validation_data=(test_data, test_labels),
    callbacks=[early_stop],
)


In [ ]:
pathlib.Path("models").mkdir(exist_ok=True)
model_path = pathlib.Path("models") / "pattern_model.h5"
model.save(model_path)
print("saved", model_path)


In [ ]:
def export_neural_model(model):
    layers = {}
    for layer in model.layers:
        weights = layer.get_weights()
        if len(weights) == 2:
            w, b = weights
            layers[layer.name] = {"w": w.tolist(), "b": b.tolist()}
    return {
        "format": "othello-pattern-nn-v1",
        "created_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "patterns": {"diagonal8": diagonal8_idx, "edge2X": edge_2x_idx, "triangle": triangle_idx},
        "layers": layers,
        "additional_features": "fallback_additional_params" if not USE_EVALUATE_OUT else "evaluate.out",
    }

NEURAL_MODEL = export_neural_model(model)
text = "NEURAL_MODEL = " + repr(NEURAL_MODEL)
path = pathlib.Path("neural_model_for_submission.py")
path.write_text(text + "\n", encoding="utf-8")
print("written", path)
print(text[:4000])
if len(text) > 4000:
    print("... preview only. neural_model_for_submission.py contains the full data.")